# Deliverable 3 — Categoricals & Relationships deep-dive

**Area:** categoricals and relationships (cardinality, encoding, linear vs non-linear, interactions).
This area was assigned to Aditi Walvekar; it was taken over by the lead (Akshith Reddy Sheri) so the team feature set
could be finished. The team feature-set code is in `feature_set.ipynb`.

**NDA:** this notebook prints summaries only, never individual rows. Clear all outputs before committing, and
do not paste exact counts into markdown.

## 1. Load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 4)

TARGET = 'OD_PAX'
RANDOM_STATE = 42
TEST_MONTHS = 12          # same hold-out as the team validation scheme: last 12 departure months

# Find the repo root (works whether Jupyter starts in the repo or in this folder)
repo_root = next((f for f in [Path.cwd(), *Path.cwd().parents]
                  if (f / 'DATA.md').is_file() and (f / 'data').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

def find_file(name):
    for folder in [repo_root / 'data' / 'raw', repo_root / 'data']:
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f'Put {name} in data/raw/ (git-ignored).')

pax = pd.read_csv(find_file('AA_PAX_FORECASTING.csv'))
try:
    airports = pd.read_csv(find_file('AA_AIRPORT_INFO.csv'))
except FileNotFoundError:
    airports = None
    print('Airport reference file not found -- country columns will be UNKNOWN.')

required = {'DEP_YEAR', 'DEP_MONTH', 'ORG_AIRPRT', 'DES_AIRPRT', TARGET}
missing_cols = required - set(pax.columns)
if missing_cols:
    raise ValueError(f'Passenger file is missing expected columns: {sorted(missing_cols)}')
print('Files loaded locally. No individual records are displayed in this notebook.')

## 2. Apply the agreed cleaning

Same steps as `feature_set.ipynb`: drop the leak / timestamp / index columns, one row per route per month,
and attach each airport's country from the airport table.

In [ ]:
# Agreed in D3 hand-offs (Sahith): these are a leak, a load timestamp and a row index -> drop.
DROP_COLS = ['OD_REV_USD', 'SF_LOAD_TMS', 'Unnamed: 0']
pax = pax.drop(columns=[c for c in DROP_COLS if c in pax.columns])

pax[TARGET] = pd.to_numeric(pax[TARGET], errors='coerce')
pax['period'] = pd.to_datetime(dict(year=pax['DEP_YEAR'], month=pax['DEP_MONTH'], day=1), errors='coerce')
bad = pax['period'].isna() | pax[TARGET].isna()
print(f'rows dropped for an unreadable date or target: {bad.mean():.2%}')

# One row per route per month (duplicates, if any, are summed)
rm = pax[~bad].groupby(['ORG_AIRPRT', 'DES_AIRPRT', 'period'], as_index=False)[TARGET].sum()
rm['ORG_AIRPRT'] = rm['ORG_AIRPRT'].astype(str)
rm['DES_AIRPRT'] = rm['DES_AIRPRT'].astype(str)
rm['route'] = rm['ORG_AIRPRT'] + '-' + rm['DES_AIRPRT']
rm['DEP_YEAR'], rm['DEP_MONTH'] = rm['period'].dt.year, rm['period'].dt.month

# Country of each airport, from the airport reference table
if airports is not None and {'AIRPRT_CD', 'CNTRY_CD'} <= set(airports.columns):
    country = (airports.assign(AIRPRT_CD=airports['AIRPRT_CD'].astype(str))
               .drop_duplicates('AIRPRT_CD').set_index('AIRPRT_CD')['CNTRY_CD'].astype(str))
else:
    country = pd.Series(dtype=str)
for side in ['ORG', 'DES']:
    rm[f'{side}_CNTRY_CD'] = rm[f'{side}_AIRPRT'].map(country).fillna('UNKNOWN')
known = (rm['ORG_CNTRY_CD'] != 'UNKNOWN') & (rm['DES_CNTRY_CD'] != 'UNKNOWN')
rm['is_international'] = (rm['ORG_CNTRY_CD'] != rm['DES_CNTRY_CD']).astype(float).where(known)
print(f'route-months with an airport missing from the reference table: {(~known).mean():.2%}')

## 3. Cardinality — how many distinct values, and how concentrated

High cardinality decides the encoding: one-hot is fine for a handful of values, but not for hundreds of airports
or thousands of routes. Concentration matters too — if a small share of routes carries most passengers, the rare
routes are mostly noise for any per-route encoding.

In [ ]:
CAT_COLS = ['ORG_AIRPRT', 'DES_AIRPRT', 'route', 'ORG_CNTRY_CD', 'DES_CNTRY_CD']
n_months = rm['period'].nunique()

def values_for_share(col, share=0.8):
    s = rm.groupby(col)[TARGET].sum().sort_values(ascending=False)
    return int((s.cumsum() / s.sum() < share).sum() + 1)

months_seen = {c: rm.groupby(c)['period'].nunique() for c in CAT_COLS}
card = pd.DataFrame({
    'distinct values': rm[CAT_COLS].nunique(),
    'values carrying 80% of passengers': [values_for_share(c) for c in CAT_COLS],
    'share of values seen in < 12 months': [(months_seen[c] < 12).mean().round(3) for c in CAT_COLS],
})
display(card)

# Concentration curve for routes: share of passengers vs share of routes (largest first)
route_tot = rm.groupby('route')[TARGET].sum().sort_values(ascending=False)
curve = route_tot.cumsum() / route_tot.sum()
plt.plot(np.arange(1, len(curve) + 1) / len(curve), curve.values)
plt.xlabel('share of routes (largest first)'); plt.ylabel('share of all passengers')
plt.title('How concentrated passengers are across routes'); plt.grid(alpha=.3); plt.show()

**Therefore we should** *(fill in after running — words only, rounded)*: e.g. "airports and routes are
high-cardinality and passengers are concentrated in a minority of routes, so one-hot on `route` is not practical;
trees get airports as native categoricals, linear models need grouped one-hot or target encoding."

## 4. New categories over time — what the model meets in the test window

With a time split, the test months can contain airports or routes that never appear in training. Whatever encoder we
choose must handle unseen values (`handle_unknown`) instead of crashing or silently mis-coding them.

In [ ]:
periods = np.sort(rm['period'].unique())
cutoff = periods[-TEST_MONTHS]
train, test = rm[rm['period'] < cutoff], rm[rm['period'] >= cutoff]

unseen = []
for c in CAT_COLS:
    new = ~test[c].isin(set(train[c]))
    unseen.append({'column': c, 'share of test rows with an unseen value': round(new.mean(), 4),
                   'share of test passengers on those rows': round(test.loc[new, TARGET].sum() / test[TARGET].sum(), 4)})
display(pd.DataFrame(unseen).set_index('column'))
print(f'test window starts at {pd.Timestamp(cutoff):%Y-%m}')

**Therefore we should** *(fill in after running)*: say whether unseen routes are rare or common in the test window, and what that means for route-level encodings.

## 5. Interactions — does seasonality depend on the route?

If the month-of-year pattern is different for international and domestic routes, or for different origin airports,
then month and route **interact**. Trees pick interactions up on their own; a linear model only sees them if we build
the cross terms by hand.

In [ ]:
prof = (rm.dropna(subset=['is_international'])
        .groupby(['is_international', 'DEP_MONTH'])[TARGET].sum())
prof = prof / prof.groupby(level='is_international').transform('sum')
prof.unstack('is_international').rename(columns={0.0: 'domestic', 1.0: 'international'}).plot(marker='o', ms=3)
plt.title('Share of each group\'s passengers by departure month'); plt.xlabel('departure month')
plt.ylabel('share of group total'); plt.xticks(range(1, 13)); plt.show()

top_org = rm.groupby('ORG_AIRPRT')[TARGET].sum().nlargest(15).index
heat = rm[rm['ORG_AIRPRT'].isin(top_org)].groupby(['ORG_AIRPRT', 'DEP_MONTH'])[TARGET].sum().unstack()
heat = heat.div(heat.sum(axis=1), axis=0)
plt.imshow(heat.values, aspect='auto', cmap='viridis'); plt.colorbar(label='share of airport\'s passengers')
plt.yticks([]); plt.xticks(range(12), range(1, 13)); plt.xlabel('departure month')
plt.title('Month profile for the 15 busiest origin airports (rows unlabeled on purpose)'); plt.show()
print(f'spread of month shares across these airports (mean std per month): {heat.std().mean():.4f}')

**Therefore we should** *(fill in after running)*: e.g. "the month pattern differs between domestic and international routes and across airports, so month x route interactions matter — another point for trees."

## 6. Linear or not — passengers vs last year's passengers

The strongest planned feature is the same route's passengers in the same month last year. If `log1p(OD_PAX)` vs
`log1p(last year)` is a straight line, a linear model on the log scale is reasonable; if it bends, trees win.

In [ ]:
ly = rm[['ORG_AIRPRT', 'DES_AIRPRT', 'period', TARGET]].copy()
ly['period'] = ly['period'] + pd.DateOffset(years=1)
rm = rm.merge(ly.rename(columns={TARGET: 'pax_same_month_last_year'}),
              on=['ORG_AIRPRT', 'DES_AIRPRT', 'period'], how='left')
pair = rm.dropna(subset=['pax_same_month_last_year'])
lx, lyv = np.log1p(pair['pax_same_month_last_year'].clip(lower=0)), np.log1p(pair[TARGET].clip(lower=0))
print(f'correlation raw scale: {pair["pax_same_month_last_year"].corr(pair[TARGET]):.3f} | '
      f'log scale: {lx.corr(lyv):.3f} | spearman: {pair["pax_same_month_last_year"].corr(pair[TARGET], method="spearman"):.3f}')

bins = pd.qcut(lx, 20, duplicates='drop')
binned = pd.DataFrame({'x': lx, 'y': lyv}).groupby(bins, observed=True).mean()
plt.plot(binned['x'], binned['y'], marker='o', label='binned mean')
lims = [binned.min().min(), binned.max().max()]
plt.plot(lims, lims, ls='--', color='grey', label='y = x')
plt.xlabel('log1p(passengers same month last year)'); plt.ylabel('log1p(passengers)')
plt.title('Is the main relationship linear on the log scale?'); plt.legend(); plt.show()

**Therefore we should** *(fill in after running)*: straight line or bend? What does that say about Ridge on `log1p` vs trees?

## 7. Encoding options compared on the honest time split

Same features, same time split (last 12 months held out), four ways to feed the categoricals:

| | Model | Categorical encoding |
|---|---|---|
| A | HistGradientBoosting (tree) | airports as native categoricals (ordinal codes, rare airports grouped) |
| B | Ridge on `log1p` | one-hot on airports and countries, rare values grouped |
| C | Ridge on `log1p` | target encoding of airports and route, **fit on training months only** (inside the pipeline) |
| D | Ridge on `log1p` | **wrong on purpose:** route average computed on *all* months, test included |

D is there to show why encoders must be fit inside the pipeline: it borrows the answer from the test months.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

rm['month_sin'] = np.sin(2 * np.pi * rm['DEP_MONTH'] / 12)
rm['month_cos'] = np.cos(2 * np.pi * rm['DEP_MONTH'] / 12)
rm['months_since_start'] = (rm['period'].dt.year - rm['period'].min().year) * 12 + rm['period'].dt.month - rm['period'].min().month
rm['log_pax_last_year'] = np.log1p(rm['pax_same_month_last_year'].clip(lower=0))
rm['route_seen_last_year'] = rm['pax_same_month_last_year'].notna().astype(int)

NUM = ['month_sin', 'month_cos', 'months_since_start', 'log_pax_last_year', 'route_seen_last_year', 'is_international']
AIRPORT_CATS = ['ORG_AIRPRT', 'DES_AIRPRT']
COUNTRY_CATS = ['ORG_CNTRY_CD', 'DES_CNTRY_CD']
tr, te = rm['period'] < cutoff, rm['period'] >= cutoff
y_tr, y_te = rm.loc[tr, TARGET], rm.loc[te, TARGET]
num_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())

def tree_model():
    cats = AIRPORT_CATS + COUNTRY_CATS
    prep = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                                                     encoded_missing_value=-1, max_categories=250), cats)],
                             remainder='passthrough', verbose_feature_names_out=False)
    return Pipeline([('prep', prep), ('m', HistGradientBoostingRegressor(
        categorical_features=list(range(len(cats))), random_state=RANDOM_STATE))])

def ridge(prep):
    return Pipeline([('prep', prep), ('m', Ridge(alpha=1.0))])

onehot = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=24), AIRPORT_CATS + COUNTRY_CATS),
    ('num', num_pipe, NUM)])
target_enc = ColumnTransformer([
    ('te', TargetEncoder(target_type='continuous'), AIRPORT_CATS + ['route']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), COUNTRY_CATS),
    ('num', num_pipe, NUM)])

def wape(y, p):
    return np.abs(y - p).sum() / np.abs(y).sum()

results = {}
X_cols = AIRPORT_CATS + COUNTRY_CATS + NUM
m = tree_model().fit(rm.loc[tr, X_cols], y_tr)
results['A  tree, native categoricals'] = m.predict(rm.loc[te, X_cols])
for name, prep, cols in [('B  Ridge log1p, one-hot', onehot, AIRPORT_CATS + COUNTRY_CATS + NUM),
                         ('C  Ridge log1p, target encoding (train only)', target_enc, AIRPORT_CATS + ['route'] + COUNTRY_CATS + NUM)]:
    m = ridge(prep).fit(rm.loc[tr, cols], np.log1p(y_tr.clip(lower=0)))
    results[name] = np.expm1(m.predict(rm.loc[te, cols])).clip(min=0)

# D: the leaky version -- route mean of log1p(target) over ALL months, including the test months
leaky = rm.assign(route_mean_all=np.log1p(rm[TARGET].clip(lower=0)).groupby(rm['route']).transform('mean'))
leaky_prep = ColumnTransformer([('num', num_pipe, NUM + ['route_mean_all'])])
m = ridge(leaky_prep).fit(leaky.loc[tr], np.log1p(y_tr.clip(lower=0)))
results['D  Ridge log1p, route mean from ALL months (leaky)'] = np.expm1(m.predict(leaky.loc[te])).clip(min=0)

seen = te & rm['pax_same_month_last_year'].notna()
table = pd.DataFrame({name: {'MAE': mean_absolute_error(y_te, p), 'WAPE': wape(y_te, p)}
                      for name, p in results.items()}).T
table.loc['same month last year (rows that have it)'] = [
    mean_absolute_error(rm.loc[seen, TARGET], rm.loc[seen, 'pax_same_month_last_year']),
    wape(rm.loc[seen, TARGET], rm.loc[seen, 'pax_same_month_last_year'])]
display(table.round(3))
print('Lower is better. Any edge D has over C is borrowed from the test months -- it could not be repeated in production.')

**Therefore we should** *(fill in after running)*: which encoding works best for the linear model (B or C),
how far the tree is from the linear options, and whether D looked suspiciously good.

## 8. Hand-off — encoding plan for the team feature set

Decisions that do not depend on the numbers above are filled in. Fill in the *(after running)* cells in words.

| Column / feature | Decision | Encoding | Reason |
|---|---|---|---|
| `ORG_AIRPRT`, `DES_AIRPRT` | keep | trees: native categorical (ordinal codes, rare airports grouped, unknown → missing); linear: *(B or C, after running)* | high cardinality; route identity drives volume |
| `route` (`ORG-DES`) | engineer | trees: not needed (the tree combines origin and destination); linear: target encoding fit inside the pipeline on training months only | too many values for one-hot |
| `ORG_CNTRY_CD`, `DES_CNTRY_CD` | engineer (airport table) | one-hot (low cardinality); `UNKNOWN` if the airport is not in the reference table | domestic vs international demand differs |
| `is_international` | engineer | 0/1, missing if either country is unknown | simple flag for the seasonality difference in section 5 |
| `STATE_PROVNC_CD`, `CITY_METRO_IATA_CD`, runway, elevation | not used for now | — | team chose country + international flag only; can revisit in D4 |
| any target or route-mean encoding | rule | always fit inside the pipeline on training months | section 7, model D, shows the leak otherwise |

## 9. Model-family view

*(fill in after running, in words)*: e.g. "high-cardinality categoricals, unseen routes in the test window, and
month x route interactions all point to gradient-boosted trees, with Ridge on `log1p` as a simpler comparison."

### Before committing
Kernel → Restart & Clear Output, then save. `git status` must show only notebooks.